In [1]:
# =========================
# DAG Validation (for our saved graphs)
# =========================
import os
import networkx as nx
from typing import List, Tuple, Optional, Dict, Any


def _load_graph(path: str) -> nx.DiGraph:
    ext = os.path.splitext(path)[1].lower()
    if ext == ".gexf":
        G = nx.read_gexf(path)
    elif ext in [".graphml", ".xml"]:
        G = nx.read_graphml(path)
    else:
        raise ValueError(f"Unsupported graph format: {ext}. Use .gexf or .graphml")
    return G


def _get_edge_abs_attr(G: nx.DiGraph, u: Any, v: Any, prefer: Tuple[str, ...] = ("weight", "score")) -> Optional[float]:
    """
    edge attribute 중 prefer 순서대로 찾아 |value| 반환.
    없으면 None.
    """
    data = G.get_edge_data(u, v, default={}) or {}
    for k in prefer:
        if k in data:
            try:
                return abs(float(data[k]))
            except Exception:
                return None
    return None


def validate_dag_graph(
    graph_path: str,
    max_cycles: int = 3,
    show_bidirectional: int = 10,
    attr_prefer: Tuple[str, ...] = ("weight", "score"),
    show_cycle_min_edge: bool = True
) -> None:
    """
    - .gexf / .graphml 그래프가 DAG인지 검증
    - cycle 존재 시 예시 출력
    - 양방향 edge 여부 점검(중복 제거)
    - (옵션) cycle마다 가장 작은 |weight|/|score| edge도 출력
    """
    print("=" * 80)
    print(f"[DAG CHECK] Loading graph: {graph_path}")

    G = _load_graph(graph_path)

    # directed 확인
    is_directed = G.is_directed()
    print(f"[INFO] directed={is_directed}, nodes={G.number_of_nodes()}, edges={G.number_of_edges()}")

    if not is_directed:
        print("[WARN] graph is not directed. DAG check requires a directed graph.")
        print("=" * 80)
        return

    # DAG 여부
    is_dag = nx.is_directed_acyclic_graph(G)
    print(f"[CHECK] is_directed_acyclic_graph: {is_dag}")

    # 양방향(edge conflict) 검사 (u<->v 중복 제거)
    bidirectional_pairs: List[Tuple[Any, Any]] = []
    seen = set()
    for u, v in G.edges():
        if G.has_edge(v, u):
            key = tuple(sorted([str(u), str(v)]))
            if key not in seen:
                seen.add(key)
                bidirectional_pairs.append((u, v))

    if bidirectional_pairs:
        print(f"[WARN] bidirectional edges detected (unique_pairs={len(bidirectional_pairs)}):")
        for u, v in bidirectional_pairs[:show_bidirectional]:
            uv = G.get_edge_data(u, v, default={}) or {}
            vu = G.get_edge_data(v, u, default={}) or {}
            print(f"  - {u} <-> {v} | {u}->{v} attrs={uv} | {v}->{u} attrs={vu}")
        if len(bidirectional_pairs) > show_bidirectional:
            print(f"  ... ({len(bidirectional_pairs) - show_bidirectional} more omitted)")
    else:
        print("[OK] no bidirectional edges")

    # cycle 검사
    if not is_dag:
        print("[ERROR] graph contains cycles")
        cycles = list(nx.simple_cycles(G))
        print(f"[INFO] number of cycles found: {len(cycles)}")

        for idx, cyc in enumerate(cycles[:max_cycles]):
            print(f"  cycle[{idx + 1}] length={len(cyc)}:")
            print("   -> " + " -> ".join([str(x) for x in cyc + [cyc[0]]]))

            if show_cycle_min_edge:
                # cycle edge 중 min |attr| 찾기
                min_edge = None
                min_val = None
                for a, b in zip(cyc, cyc[1:] + [cyc[0]]):
                    val = _get_edge_abs_attr(G, a, b, prefer=attr_prefer)
                    if val is None:
                        continue
                    if (min_val is None) or (val < min_val):
                        min_val = val
                        min_edge = (a, b)

                if min_edge is not None:
                    a, b = min_edge
                    data = G.get_edge_data(a, b, default={}) or {}
                    print(f"    [HINT] smallest |{attr_prefer}| edge in this cycle: {a}->{b}, attrs={data}")
                else:
                    print("    [HINT] could not find numeric edge attributes for cycle edges")

        if len(cycles) > max_cycles:
            print(f"  ... ({len(cycles) - max_cycles} more cycles omitted)")
    else:
        print("[OK] graph is a valid DAG")

    print("=" * 80)


In [2]:
validate_dag_graph("./graph_NOTEARS.gexf")
validate_dag_graph("./graph_PC.gexf")
validate_dag_graph("./graph_GES.gexf")
validate_dag_graph("./graph_GOLEM.gexf")

[DAG CHECK] Loading graph: ./graph_NOTEARS.gexf
[INFO] directed=True, nodes=13, edges=9
[CHECK] is_directed_acyclic_graph: True
[OK] no bidirectional edges
[OK] graph is a valid DAG
[DAG CHECK] Loading graph: ./graph_PC.gexf
[INFO] directed=True, nodes=13, edges=24
[CHECK] is_directed_acyclic_graph: True
[OK] no bidirectional edges
[OK] graph is a valid DAG
[DAG CHECK] Loading graph: ./graph_GES.gexf
[INFO] directed=True, nodes=14, edges=55
[CHECK] is_directed_acyclic_graph: True
[OK] no bidirectional edges
[OK] graph is a valid DAG
[DAG CHECK] Loading graph: ./graph_GOLEM.gexf
[INFO] directed=True, nodes=13, edges=11
[CHECK] is_directed_acyclic_graph: True
[OK] no bidirectional edges
[OK] graph is a valid DAG


In [ ]:
import os
import re
import shutil
import subprocess
from typing import Dict, Optional, List, Tuple, Callable

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt


# =========================
# Settings
# =========================
BASE_DIR = "."
OUT_BASE = os.path.join(BASE_DIR, "layouts_all")

PATHS = {
    "NOTEARS": os.path.join(BASE_DIR, "graph_NOTEARS.gexf"),
    "GOLEM": os.path.join(BASE_DIR, "graph_GOLEM.gexf"),
    "PC": os.path.join(BASE_DIR, "graph_PC.gexf"),
    "GES": os.path.join(BASE_DIR, "graph_GES.gexf"),
}

FIGSIZE = (28, 20)
DPI = 300

ARROWSIZE = 32
EDGE_W_MIN = 1.2
EDGE_W_MAX = 5.0

TOPK_EDGE_LABELS = 30
EPS = 0.0
TOP_EDGES = None  # 예: 200

LABEL_BBOX = True
EDGE_LABEL_BBOX = True

# dot(계층형) 방향: TB(위->아래) / LR(좌->우)
DOT_RANKDIR = "TB"  # "LR" 추천도 가능

LAYOUTS = [
    "spring",
    "kamada_kawai",
    "spectral",
    "circular",
    "shell",
    "random",
    "spiral",
    "multipartite_topo",
    "bfs_layers",
    "radial_layers",
    "spring_tight",
    "spring_loose",
    "graphviz_dot",
    "graphviz_neato",
    "graphviz_fdp",
    "graphviz_sfdp",
    "graphviz_circo",
    "graphviz_twopi",
    "planar_or_fallback",
]

GRAPHVIZ_BIN_CANDIDATES = [
    r"C:\Program Files\Graphviz\bin",
    r"C:\Program Files (x86)\Graphviz\bin",
]


# =========================
# Graphviz PATH bootstrap
# =========================
def ensure_graphviz_on_path() -> None:
    if shutil.which("dot") is not None:
        return
    for cand in GRAPHVIZ_BIN_CANDIDATES:
        dot_exe = os.path.join(cand, "dot.exe")
        if os.path.exists(dot_exe):
            os.environ["PATH"] = cand + os.pathsep + os.environ.get("PATH", "")
            break


def has_graphviz() -> bool:
    return shutil.which("dot") is not None


# =========================
# Utils
# =========================
def ensure_dir(p: str) -> None:
    os.makedirs(p, exist_ok=True)


def _as_float(x) -> float:
    try:
        return float(x)
    except Exception:
        return 0.0


def normalize_pos(pos: Dict[str, Tuple[float, float]]) -> Dict[str, Tuple[float, float]]:
    """좌표 스케일 차이(프로그램별)를 줄이기 위해 평균0/표준편차1로 정규화"""
    if not pos:
        return pos
    xs = np.array([p[0] for p in pos.values()], dtype=float)
    ys = np.array([p[1] for p in pos.values()], dtype=float)
    xmu, ymu = float(xs.mean()), float(ys.mean())
    xsd, ysd = float(xs.std()), float(ys.std())
    if xsd < 1e-12:
        xsd = 1.0
    if ysd < 1e-12:
        ysd = 1.0
    return {str(k): (float((x - xmu) / xsd), float((y - ymu) / ysd)) for k, (x, y) in pos.items()}


def load_gexf_as_digraph(path: str) -> nx.DiGraph:
    """MultiDiGraph 방지 + 속성 보존 + 노드명 str 통일"""
    G0 = nx.read_gexf(path)
    G = nx.DiGraph()

    for n, data in G0.nodes(data=True):
        G.add_node(str(n), **data)

    if isinstance(G0, nx.MultiDiGraph):
        for u, v, k, data in G0.edges(keys=True, data=True):
            u2, v2 = str(u), str(v)
            nd = dict(data)
            for attr in ["weight", "score", "w"]:
                if attr in nd:
                    nd[attr] = _as_float(nd.get(attr, 0.0))

            if not G.has_edge(u2, v2):
                G.add_edge(u2, v2, **nd)
            else:
                for attr in ["weight", "score", "w"]:
                    if attr in nd:
                        newv = _as_float(nd.get(attr, 0.0))
                        oldv = _as_float(G[u2][v2].get(attr, 0.0))
                        if abs(newv) > abs(oldv):
                            G[u2][v2][attr] = newv
    else:
        for u, v, data in G0.edges(data=True):
            u2, v2 = str(u), str(v)
            nd = dict(data)
            for attr in ["weight", "score", "w"]:
                if attr in nd:
                    nd[attr] = _as_float(nd.get(attr, 0.0))
            G.add_edge(u2, v2, **nd)

    G.remove_edges_from(list(nx.selfloop_edges(G)))
    return G


def pick_edge_attr_for_alg(alg: str, G: nx.DiGraph) -> Optional[str]:
    """weight 우선, 없으면 score, w"""
    candidates = ["weight", "score", "w"]
    edges = list(G.edges())
    if not edges:
        return None
    for attr in candidates:
        for u, v in edges[: min(200, len(edges))]:
            if attr in G[u][v]:
                return attr
    return None


def filter_edges(G: nx.DiGraph, edge_attr: Optional[str]) -> nx.DiGraph:
    """|w|<=EPS 제거 + (선택) TOP_EDGES로 상위 |w|만 유지"""
    H = nx.DiGraph()
    H.add_nodes_from(G.nodes(data=True))

    if edge_attr is None:
        for u, v, data in G.edges(data=True):
            if u != v:
                H.add_edge(u, v, **data)
        return H

    tmp = []
    for u, v, data in G.edges(data=True):
        if u == v:
            continue
        val = _as_float(data.get(edge_attr, 0.0))
        if abs(val) <= EPS:
            continue
        tmp.append((u, v, val, data))

    if TOP_EDGES is not None and len(tmp) > TOP_EDGES:
        tmp = sorted(tmp, key=lambda x: abs(x[2]), reverse=True)[:TOP_EDGES]

    for u, v, val, data in tmp:
        H.add_edge(u, v, **data)

    return H


def auto_style_params(G: nx.DiGraph) -> Tuple[int, int, int, int]:
    """
    노드 수에 따라 node size / label font를 자동 조정
    return: node_min, node_max, font_node, font_edge
    """
    n = G.number_of_nodes()
    if n <= 15:
        return 1800, 5200, 16, 12
    if n <= 30:
        return 1400, 4200, 14, 11
    if n <= 60:
        return 1050, 3300, 12, 10
    if n <= 120:
        return 850, 2600, 10, 9
    if n <= 200:
        return 650, 2100, 9, 8
    return 520, 1700, 8, 7


def compute_node_sizes(G: nx.DiGraph, node_min: int, node_max: int) -> List[float]:
    """차수 기반 node size"""
    deg = {n: (G.in_degree(n) + G.out_degree(n)) for n in G.nodes()}
    if not deg:
        return []
    vals = np.array(list(deg.values()), dtype=float)
    vmin, vmax = float(vals.min()), float(vals.max())
    if vmax - vmin < 1e-12:
        return [(node_min + node_max) / 2 for _ in G.nodes()]
    out = []
    for n in G.nodes():
        t = (deg[n] - vmin) / (vmax - vmin)
        out.append(node_min + t * (node_max - node_min))
    return out


def compute_edge_widths(values: List[float]) -> List[float]:
    """|w| -> 굵기"""
    if not values:
        return []
    absvals = np.array([abs(v) for v in values], dtype=float)
    vmin, vmax = float(absvals.min()), float(absvals.max())
    if vmax - vmin < 1e-12:
        return [2.2 for _ in values]
    out = []
    for a in absvals:
        t = (a - vmin) / (vmax - vmin)
        t = float(np.clip(t, 0.0, 1.0))
        t = np.sqrt(t)
        out.append(EDGE_W_MIN + t * (EDGE_W_MAX - EDGE_W_MIN))
    return out


def connected_components_positions(
    UG: nx.Graph,
    layout_fn: Callable[[nx.Graph], Dict]
) -> Dict[str, Tuple[float, float]]:
    """컴포넌트별로 layout 후 x-offset으로 겹침 방지"""
    comps = list(nx.connected_components(UG))
    if not comps:
        return {}

    all_pos: Dict[str, Tuple[float, float]] = {}
    x_offset = 0.0
    for comp in sorted(comps, key=len, reverse=True):
        sub = UG.subgraph(comp).copy()
        pos_c = layout_fn(sub)

        xs = [pos_c[n][0] for n in sub.nodes()] if sub.number_of_nodes() else [0.0]
        minx, maxx = float(min(xs)), float(max(xs))
        width = (maxx - minx) if (maxx - minx) > 1e-9 else 1.0

        for n, (x, y) in pos_c.items():
            all_pos[str(n)] = (float(x + x_offset), float(y))
        x_offset += width * 2.2

    return all_pos


def topo_levels(G: nx.DiGraph) -> Dict[str, int]:
    """DAG depth 레벨. DAG 아니면 in-degree fallback."""
    try:
        order = list(nx.topological_sort(G))
    except Exception:
        return {n: int(G.in_degree(n)) for n in G.nodes()}
    level = {n: 0 for n in G.nodes()}
    for v in order:
        preds = list(G.predecessors(v))
        if preds:
            level[v] = 1 + max(level[p] for p in preds)
    return level


def bfs_layers(G: nx.DiGraph) -> Dict[str, int]:
    """진입차수 0 노드들을 루트로 BFS 레벨"""
    roots = [n for n in G.nodes() if G.in_degree(n) == 0]
    if not roots:
        return topo_levels(G)
    UG = G.to_undirected()
    dist = {r: 0 for r in roots}
    from collections import deque
    q = deque(roots)
    while q:
        u = q.popleft()
        for v in UG.neighbors(u):
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return {n: int(dist.get(n, 0)) for n in G.nodes()}


# =========================
# Graphviz CLI layout (stable: -Tplain)
# =========================
_PLAIN_NODE_RE = re.compile(
    r'^node\s+(?P<name>"[^"]+"|\S+)\s+(?P<x>-?\d+(\.\d+)?)\s+(?P<y>-?\d+(\.\d+)?)\s+',
    re.M
)

def _unquote(s: str) -> str:
    s = s.strip()
    if len(s) >= 2 and s[0] == '"' and s[-1] == '"':
        return s[1:-1]
    return s

def graphviz_cli_layout(G: nx.DiGraph, prog: str) -> Dict[str, Tuple[float, float]]:
    """
    Graphviz 실행(dot/neato/...) -> -Tplain 출력에서 좌표 파싱
    - pydot / graphviz_layout() 사용 안 함
    - 파싱이 안정적 (node 라인이 항상 x,y 포함)
    """
    ensure_graphviz_on_path()
    if not has_graphviz():
        raise RuntimeError("dot not found in PATH for this python process.")

    # DOT 만들기 (최소 속성)
    lines = []
    lines.append("digraph G {")
    if prog == "dot":
        lines.append(f'graph [rankdir="{DOT_RANKDIR}", nodesep="0.30", ranksep="0.45"];')
    else:
        # neato/fdp/sfdp 쪽은 겹침 완화
        lines.append('graph [overlap="prism"];')
    lines.append('node [shape="ellipse"];')

    for n in G.nodes():
        lines.append(f'"{n}";')
    for u, v in G.edges():
        lines.append(f'"{u}" -> "{v}";')

    lines.append("}")
    dot_in = "\n".join(lines).encode("utf-8")

    # 핵심: -Tplain (좌표를 node 라인으로 제공)
    cmd = [prog, "-Tplain"]
    p = subprocess.run(cmd, input=dot_in, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if p.returncode != 0:
        raise RuntimeError(f"Graphviz failed: {p.stderr.decode('utf-8', errors='ignore')[:400]}")

    out = p.stdout.decode("utf-8", errors="ignore")

    pos_raw: Dict[str, Tuple[float, float]] = {}
    for m in _PLAIN_NODE_RE.finditer(out):
        name = _unquote(m.group("name"))
        x = float(m.group("x"))
        y = float(m.group("y"))
        pos_raw[name] = (x, y)

    # 이름 매칭 보정(혹시 plain이 축약/가공한 경우 대비: 여기선 그대로 매칭만)
    pos: Dict[str, Tuple[float, float]] = {}
    for n in G.nodes():
        ns = str(n)
        if ns in pos_raw:
            pos[ns] = pos_raw[ns]

    if not pos:
        # 디버깅용으로 일부 출력
        head = "\n".join(out.splitlines()[:40])
        raise RuntimeError("Graphviz succeeded but no node positions were parsed from -Tplain output.\n"
                           f"plain head:\n{head}")

    print(f"[GRAPHVIZ OK] prog={prog} parsed_pos={len(pos)}/{G.number_of_nodes()}")
    return normalize_pos(pos)


# =========================
# Layout switch
# =========================
def safe_layout(G: nx.DiGraph, layout_name: str) -> Dict[str, Tuple[float, float]]:
    """
    레이아웃 계산에서 weight/score 사용 안 함.
    graphviz_*는 가능하면 CLI로 수행, 실패하면 spring fallback.
    """
    UG = G.to_undirected()

    def spring_default(H): return nx.spring_layout(H, seed=42)
    def spring_tight(H): return nx.spring_layout(H, seed=42, k=0.08, iterations=300)
    def spring_loose(H): return nx.spring_layout(H, seed=42, k=0.35, iterations=200)
    def kk(H): return nx.kamada_kawai_layout(H)
    def spectral(H): return nx.spectral_layout(H)
    def circular(H): return nx.circular_layout(H)
    def shell(H): return nx.shell_layout(H)
    def random(H): return nx.random_layout(H, seed=42)
    def spiral(H): return nx.spiral_layout(H)

    def multipartite_topo(_):
        levels = topo_levels(G)
        for n, lv in levels.items():
            G.nodes[n]["subset"] = int(lv)
        return nx.multipartite_layout(G, subset_key="subset")

    def bfs_layers_layout(_):
        levels = bfs_layers(G)
        for n, lv in levels.items():
            G.nodes[n]["subset"] = int(lv)
        return nx.multipartite_layout(G, subset_key="subset")

    def radial_layers_layout(_):
        levels = topo_levels(G)
        max_lv = max(levels.values()) if levels else 0
        shells = []
        for lv in range(max_lv + 1):
            shells.append([n for n in G.nodes() if levels.get(n, 0) == lv])
        shells = [s for s in shells if s]
        return nx.shell_layout(UG, nlist=shells)

    def planar_or_fallback(H):
        try:
            pos = nx.planar_layout(H)
            return {str(k): (float(v[0]), float(v[1])) for k, v in pos.items()}
        except Exception:
            return spring_default(H)

    try:
        if layout_name == "spring":
            return connected_components_positions(UG, spring_default)
        if layout_name == "spring_tight":
            return connected_components_positions(UG, spring_tight)
        if layout_name == "spring_loose":
            return connected_components_positions(UG, spring_loose)
        if layout_name == "kamada_kawai":
            return connected_components_positions(UG, kk)
        if layout_name == "spectral":
            return connected_components_positions(UG, spectral)
        if layout_name == "circular":
            return connected_components_positions(UG, circular)
        if layout_name == "shell":
            return connected_components_positions(UG, shell)
        if layout_name == "random":
            return connected_components_positions(UG, random)
        if layout_name == "spiral":
            return connected_components_positions(UG, spiral)
        if layout_name == "multipartite_topo":
            return multipartite_topo(UG)
        if layout_name == "bfs_layers":
            return bfs_layers_layout(UG)
        if layout_name == "radial_layers":
            return radial_layers_layout(UG)
        if layout_name == "planar_or_fallback":
            return connected_components_positions(UG, planar_or_fallback)

        if layout_name.startswith("graphviz_"):
            prog = layout_name.replace("graphviz_", "")

            # dot은 DAG 계층형에 좋지만, 비-DAG면 가끔 이상해질 수 있어 안전장치
            if prog == "dot" and not nx.is_directed_acyclic_graph(G):
                prog = "sfdp"

            try:
                return graphviz_cli_layout(G, prog=prog)
            except Exception as e:
                print(f"[GRAPHVIZ FAIL] layout={layout_name} reason={repr(e)} -> fallback to spring")
                return connected_components_positions(UG, spring_default)

        return connected_components_positions(UG, spring_default)
    except Exception:
        return connected_components_positions(UG, spring_default)


# =========================
# Drawing
# =========================
def draw_graph(
    alg: str,
    G: nx.DiGraph,
    edge_attr: Optional[str],
    layout_name: str,
    out_dir: str
) -> None:
    ensure_dir(out_dir)

    H = filter_edges(G, edge_attr=edge_attr)
    edges = list(H.edges())
    if len(edges) == 0:
        print(f"[SKIP] {alg} | {layout_name}: no edges after filtering (edge_attr={edge_attr})")
        return

    pos = safe_layout(H, layout_name)

    # pos 누락 보정
    if len(pos) < H.number_of_nodes():
        missing = [n for n in H.nodes() if n not in pos]
        fb = nx.spring_layout(H.to_undirected(), seed=42)
        for n in missing:
            pos[n] = (float(fb[n][0]), float(fb[n][1]))

    node_min, node_max, font_node, font_edge = auto_style_params(H)

    if edge_attr is None:
        values = [0.0 for _ in edges]
        edge_colors = ["0.35" for _ in edges]
    else:
        values = [_as_float(H[u][v].get(edge_attr, 0.0)) for u, v in edges]
        # 색상 지정이 싫으면 이 줄을 한 가지 색으로 통일하세요.
        edge_colors = ["tab:blue" if v > 0 else "tab:red" if v < 0 else "0.55" for v in values]

    widths = compute_edge_widths(values)

    edge_labels = {}
    if edge_attr is not None and TOPK_EDGE_LABELS > 0:
        order = np.argsort([abs(v) for v in values])[::-1]
        for idx in order[: min(TOPK_EDGE_LABELS, len(order))]:
            u, v = edges[idx]
            edge_labels[(u, v)] = f"{values[idx]:.2f}"

    node_sizes = compute_node_sizes(H, node_min=node_min, node_max=node_max)

    title_suffix = "structure only" if edge_attr is None else edge_attr
    title = f"{alg} | {layout_name} | {title_suffix} | n={H.number_of_nodes()} e={H.number_of_edges()} | EPS={EPS} | TOP_EDGES={TOP_EDGES}"
    out_png = os.path.join(out_dir, f"{alg}__{layout_name}.png")

    plt.figure(figsize=FIGSIZE, dpi=DPI)
    plt.title(title)

    nx.draw_networkx_nodes(
        H, pos,
        node_size=node_sizes,
        node_color="white",
        edgecolors="black",
        linewidths=1.0
    )

    if LABEL_BBOX:
        nx.draw_networkx_labels(
            H, pos,
            font_size=font_node,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.72, pad=0.9)
        )
    else:
        nx.draw_networkx_labels(H, pos, font_size=font_node)

    nx.draw_networkx_edges(
    H, pos,
    edge_color=edge_colors,
    width=widths if widths else 1.6,
    arrows=True,
    arrowsize=ARROWSIZE,
    arrowstyle="-|>",          # 머리가 더 잘 보이는 스타일
    alpha=0.90,
    connectionstyle="arc3,rad=0.06",
    min_source_margin=6,       # 노드 테두리 밖으로 화살표가 나오게
    min_target_margin=10,      # target 쪽 여백(화살표 머리 공간)
)


    if edge_labels:
        if EDGE_LABEL_BBOX:
            nx.draw_networkx_edge_labels(
                H, pos,
                edge_labels=edge_labels,
                font_size=font_edge,
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.72, pad=0.35)
            )
        else:
            nx.draw_networkx_edge_labels(H, pos, edge_labels=edge_labels, font_size=font_edge)

    plt.axis("off")
    plt.tight_layout()
    plt.savefig(out_png)
    plt.close()

    print(f"[DONE] {alg} | {layout_name} -> {out_png}")


def main():
    ensure_dir(OUT_BASE)
    ensure_graphviz_on_path()

    if not has_graphviz():
        print("[WARN] Graphviz 'dot' not found in PATH for this Python process.")
        print("       graphviz_* layouts will fallback to spring.")
    else:
        print("[INFO] dot =", shutil.which("dot"))

    graphs: Dict[str, nx.DiGraph] = {}
    for alg, p in PATHS.items():
        if not os.path.exists(p):
            print(f"[SKIP] Missing: {p}")
            continue
        graphs[alg] = load_gexf_as_digraph(p)

    if not graphs:
        raise FileNotFoundError("No graphs to draw. Check BASE_DIR/PATHS.")

    for alg, G in graphs.items():
        edge_attr = pick_edge_attr_for_alg(alg, G)
        out_dir = os.path.join(OUT_BASE, alg)

        for layout_name in LAYOUTS:
            draw_graph(
                alg=alg,
                G=G,
                edge_attr=edge_attr,
                layout_name=layout_name,
                out_dir=out_dir
            )

    print(f"[ALL DONE] outputs -> {OUT_BASE}")
    print("Layouts used:", ", ".join(LAYOUTS))
    print(f"Zero-edge filter: abs(value) <= {EPS} removed (when edge_attr is not None)")


if __name__ == "__main__":
    main()


[INFO] dot = C:\Program Files (x86)\Graphviz\bin\dot.EXE
[DONE] NOTEARS | spring -> .\layouts_all\NOTEARS\NOTEARS__spring.png
[DONE] NOTEARS | kamada_kawai -> .\layouts_all\NOTEARS\NOTEARS__kamada_kawai.png
[DONE] NOTEARS | spectral -> .\layouts_all\NOTEARS\NOTEARS__spectral.png
[DONE] NOTEARS | circular -> .\layouts_all\NOTEARS\NOTEARS__circular.png
[DONE] NOTEARS | shell -> .\layouts_all\NOTEARS\NOTEARS__shell.png
[DONE] NOTEARS | random -> .\layouts_all\NOTEARS\NOTEARS__random.png
[DONE] NOTEARS | spiral -> .\layouts_all\NOTEARS\NOTEARS__spiral.png
[DONE] NOTEARS | multipartite_topo -> .\layouts_all\NOTEARS\NOTEARS__multipartite_topo.png
[DONE] NOTEARS | bfs_layers -> .\layouts_all\NOTEARS\NOTEARS__bfs_layers.png
[DONE] NOTEARS | radial_layers -> .\layouts_all\NOTEARS\NOTEARS__radial_layers.png
[DONE] NOTEARS | spring_tight -> .\layouts_all\NOTEARS\NOTEARS__spring_tight.png
[DONE] NOTEARS | spring_loose -> .\layouts_all\NOTEARS\NOTEARS__spring_loose.png
[GRAPHVIZ OK] prog=dot parsed